# Plot solute profiles

-Big differences in dissolution fractions?
-Big differences in alkalinity export?


In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from byte_util.util import all_sites, all_forcing_types
from byte_util.met_forcing_plots import profiles_to_plot, plot_profiles, plot_exchange_profiles

## Create PDF files with profiles of several species across all sites and all forcings

In [ ]:
replot = True

for site in tqdm(all_sites, desc='Iterating over sites'):
    if not replot:
        continue
    pdf_file = Path('plots') / f'ERWProfiles_{site}.pdf'

    with PdfPages(pdf_file) as pdf:
        for forcing in tqdm(all_forcing_types, leave=False, desc='Iterating over forcing types'):

            # Add forcing page
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.annotate(forcing, (0.5, 0.5), fontsize=32, ha='center', va='center')
            ax.xaxis.set_visible(False)
            ax.yaxis.set_visible(False)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            for page_no, plot_info in tqdm(profiles_to_plot.items(), leave=False, desc=f'Iterating over pages'):
                if plot_info['vars'] == ['plot_exchange_profiles']:
                    for itime in [0, -1]:
                        for scenario in ['ctrl', 'erw']:
                            fig, ax = plot_exchange_profiles(site, forcing, scenario=scenario, itime=itime)
                            fig.suptitle(f'{forcing} ({scenario}): {plot_info['title']}')
                            pdf.savefig(fig, bbox_inches='tight')
                            plt.close(fig)
                else:
                    fig, ax = plot_profiles(plot_info['vars'], site, forcing, scenario='both')
                    fig.suptitle(f'{forcing}: {plot_info['title']}')
                    pdf.savefig(fig, bbox_inches='tight')
                    plt.close(fig)

# Scratch space to check other profiles/histories

In [ ]:
from byte_util.met_forcing_plots import plot_bcvs_soi, plot_histories

site = 'Cecil'
forcing = 'monthly'
fig, ax = plot_profiles(['p_w', 'theta_a', 'q_root'], site, forcing, scenario='both')
fig.suptitle('Water content and root uptake')

In [ ]:
site = 'Cecil'
forcing = 'monthly'
scenario = 'ctrl'
plot_bcvs_soi('Cecil', 'monthly', 'ctrl', figsize=(8, 4), time_unit='d')
for page_no, plot_info in profiles_to_plot.items():
    if page_no not in [6, 8, 9]:
        continue
    vars_to_plot = plot_info['vars']
    title = plot_info['title']
    f, a = plot_histories(vars_to_plot, site, forcing, scenario, plot_cells=[396, 391, 371, 361, 351, 331],
                          time_unit='d')
    f.suptitle(plot_info['title'])